<a href="https://colab.research.google.com/github/TayBean19/intro_to_python/blob/main/ExploringVectorandRasterData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Geospatial Data in Python

What is Geospatial Data?

Geospatial data is data that has a location component — it tells us where something is on Earth. There are two primary types:

Vector Data: Represents features as points, lines, and polygons.

Example: Cities (points), Roads (lines), Countries (polygons).

Stored in formats like Shapefiles (.shp, .shx, .dbf), GeoJSON, or Geopackage (.gpkg).

Raster Data: Represents data as a grid of cells (pixels). Each cell has a value.

Example: Elevation models (DEM), satellite imagery, temperature grids.

Stored in formats like GeoTIFF (.tif) or NetCDF (.nc).

Vector Data in Python (GeoPandas)
2.1 What is a Shapefile?

A shapefile is a common (but older) format for storing vector data. It actually consists of at least 3 files with the same name but different extensions:

.shp → geometry (points, lines, polygons).

.shx → index for fast lookup.

.dbf → attributes (like a table of data).

You must keep these files together.

In [1]:
%pip install contextily

In [2]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx # For adding basemaps

Part 1: Setting Up the Environment

First, we import the core libraries. In a professional workflow, these same libraries are often used within ArcGIS Pro via Jupyter Notebooks to automate data preparation.

1. Vector Data Manipulation with GeoPandas

Vector data is handled via the GeoDataFrame. In this exercise, we will load a dataset, filter it, and perform a geometric calculation (area).

In [3]:
import geopandas as gpd

# 1. Load the Landmark Shapefile
# Students: Replace the path with your downloaded TIGER shapefile
landmarks = gpd.read_file("/tl_2025_47_pointlm.zip")

# 2. Inspection: What is inside?
print(f"Geometry Type: {landmarks.geometry.type.unique()}")
print(landmarks.columns)

# 3. Manipulation: Selecting specific types of landmarks
# Often landmarks have a 'MTFCC' code or a 'NAME' attribute
# Challenge: Filter for 'Hospital' or 'School' if the data allows
schools = landmarks[landmarks['FULLNAME'].str.contains("School", na=False)]

# 4. CRS Alignment
# Landmarks usually come in EPSG:4269 (NAD83).
# To layer them on a web map, we often need EPSG:3857.
schools = schools.to_crs(epsg=3857)

print(f"First 5 Schools:\n {schools[['FULLNAME', 'geometry']].head()}")

Geometry Type: ['Point']
Index(['STATEFP', 'ANSICODE', 'POINTID', 'FULLNAME', 'MTFCC', 'geometry'], dtype='object')
First 5 Schools:
                                    FULLNAME                          geometry
50                   Crossroads Elem School  POINT (-9515976.908 4189852.686)
55                    Lynchburg Elem School  POINT (-9615012.627 4203053.757)
57                 Moore County High School  POINT (-9613641.728 4204915.737)
323                     Lebanon High School  POINT (-9605326.384 4328478.464)
378  Woodbury 7th Day Adventist Elem School  POINT (-9581071.537 4264311.349)


**Your turn!**

Vector data is all about filtering logic and coordinate math.
Challenge 1: The Attribute Filter

Task: Instead of filtering by continent, modify the code to find landmarks related to parks.

    The Change: Locate the line hospitals = landmarks[landmarks['NAME'] == 'Hospital'].

    The Goal: Change it to find landmarks that are related to parks.

    Optional Goal: Filter the parks you get from the previous answer for parks you would actually want to walk through. :D

    Why it matters: This teaches you how to query spatial data for particular attributes or data points that you are looking for within a dataset.

In [4]:
import geopandas as gdp

landmarks = gdp.read_file("/tl_2025_47_pointlm.zip")

print(f"Geometry Type: {landmarks.geometry.type.unique()}")
print(landmarks.columns)

#filter landmarks for just parks
parks = landmarks[landmarks['FULLNAME'].str.contains("Park", na=False)]

#Exculde Trailer Parks and Cemetaries
walkable_parks = parks[~parks['FULLNAME'].str.contains("Trailer|Cmtry",case=False, na=False )]


walkable_parks = walkable_parks.to_crs(epsg=3857)

print(f"First 5 Parks:\n {walkable_parks[['FULLNAME', 'geometry']].head()}")








Geometry Type: ['Point']
Index(['STATEFP', 'ANSICODE', 'POINTID', 'FULLNAME', 'MTFCC', 'geometry'], dtype='object')
First 5 Parks:
                      FULLNAME                          geometry
13           Franklin Co Park  POINT (-9581704.054 4205852.165)
15   Estill Springs City Park  POINT (-9587000.858 4199841.993)
16        Estill Springs Park  POINT (-9588546.418 4201524.801)
190          Vonore City Park  POINT (-9378029.016 4244666.114)
191              Houston Park  POINT (-9390842.335 4234929.731)


2. Raster Data Manipulation with Rasterio

Raster data is essentially a multi-dimensional array (or matrix) of pixels. In Python, we use rasterio to interact with the metadata and pixel values of these grids (e.g., satellite imagery or elevation models).

In [ ]:
#Simple way to open a Raster (However, Colab's ram is too small so we need to do
#some partitioning or resampling to handle our file here)
import rasterio
import matplotlib.pyplot as plt

# 1. Open the file
with rasterio.open('/global_pop_2025_CN_1km_R2025A_UA_v1 (1).tif') as src:
    # 2. Read the entire dataset into one massive array
    pop_data = src.read(1)

    # 3. Calculate the total (Ignoring NoData values < 0)
    total_pop = pop_data[pop_data > 0].sum()

    # 4. Create the plot
    plt.figure(figsize=(15, 7))
    plt.imshow(pop_data, cmap='magma', vmax=50)
    plt.title(f"Global Population: {total_pop:,.0f}")
    plt.axis('off')
    plt.show()

In [ ]:
import rasterio
from rasterio.enums import Resampling
import numpy as np
import matplotlib.pyplot as plt

file_path = '/global_pop_2025_CN_1km_R2025A_UA_v1 (1).tif'

with rasterio.open(file_path) as src:
    # --- STEP 1: ACCURATE MATH (The 'Block' Method) ---
    # We calculate the total sum by reading the file in 'windows' (blocks).
    # This uses almost ZERO RAM because we never load the whole world at once.
    total_population = 0
    for ji, window in src.block_windows(1):
        block_data = src.read(1, window=window).astype(float)
        block_data[block_data < 0] = 0 # Clean NoData
        total_population += np.sum(block_data)

    # --- STEP 2: RAM-SAVING MAP (Decimation) ---
    scale_factor = 0.1
    pop_display = src.read(
        1,
        out_shape=(int(src.height * scale_factor), int(src.width * scale_factor)),
        resampling=Resampling.bilinear # 'Average' is better for population than 'Bilinear'
    ).astype(float)

    pop_display[pop_display < 0] = 0

# --- STEP 3: VISUALIZATION ---
plt.figure(figsize=(20, 10))
ax = plt.gca()
ax.set_facecolor('#000022') # Deep Navy

# Contrast stretch (VMAX=50) to make the landmasses visible
img = plt.imshow(pop_display, cmap='magma', vmax=50)

plt.colorbar(img, label='People per km²', fraction=0.02, pad=0.04)
plt.title(f"Global Population: {total_population:,.0f}", fontsize=22, pad=20)
plt.axis('off')
plt.show()

print(f"Verified Total Population: {total_population:,.0f}")

Change the above code from average to 'bilinear' what happens to our total world population. What happens? Give the number that is printed with your map in another code block below.

In [ ]:
# new number = 8,196,931,816

Bonus exercise: Open the acpcp.mon.ltm.nc file.

Opening a NetCDF (.nc) file is slightly different from opening a GeoTIFF. While GeoTIFFs are usually "flat" images, NetCDF files are multidimensional. They often contain layers for longitude, latitude, and time, allowing you to store decades of climate data in a single file.

In the Python ecosystem, Xarray is the industry standard for NetCDF because it handles these dimensions (like time) by name, making it much easier to use than raw arrays.


In [1]:
import xarray as xr
import matplotlib.pyplot as plt

# Open with decode_times=False
ds = xr.open_dataset('/apcp.mon.mean.nc', decode_times=False)

# The 'time' variable will now just be raw numbers (e.g., hours since 1800)
print(ds.time)

# 2. Inspect the contents (Variables, Dimensions, Coordinates)
print(ds)

# 3. Access the variable and calculate the mean across all time steps
# Assuming the variable is 'acpcp' (Convective Precipitation)
precip_mean = ds['acpcp'].mean(dim='time')

# 4. Plot the results
plt.figure(figsize=(12, 6))
precip_mean.plot(cmap='Blues')
plt.title("Long-term Mean Convective Precipitation")
plt.show()

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/apcp.mon.mean.nc', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

Change the color scheme used to show the magnitude of convective precipitation from blue to red. Does it look better? Why would blue be considered a better color to represent rain in most circumstances? When would you want to use red?